In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


# **Step 1 — Load Dataset**

In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
sample_submission = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/sample_submission.csv')

In [3]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Submission Shape:", sample_submission.shape)

Train Shape: (8693, 14)
Test Shape: (4277, 13)
Sample Submission Shape: (4277, 2)


# **Step 2 — Quick Dataset Preview**

In [4]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


# **Step 2 — Basic Dataset Information**

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


# **Step 3 — Statistical Summary**

In [7]:
train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


# **Step 4 — Missing Values**

In [8]:
train.isnull().sum().sort_values(ascending=False)

CryoSleep       217
ShoppingMall    208
VIP             203
HomePlanet      201
Name            200
Cabin           199
VRDeck          188
Spa             183
FoodCourt       183
Destination     182
RoomService     181
Age             179
PassengerId       0
Transported       0
dtype: int64

# **Step 5 — Target Distribution**

In [9]:
train['Transported'].value_counts()

Transported
True     4378
False    4315
Name: count, dtype: int64

In [10]:
train['Transported'].value_counts(normalize=True) * 100

Transported
True     50.362361
False    49.637639
Name: proportion, dtype: float64

# **Step 6 — Data Types Count**

In [11]:
train.dtypes.value_counts()

object     7
float64    6
bool       1
Name: count, dtype: int64

# **Step 7 — Feature Understanding**

In [12]:
train.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='object')

# **Step 8 — Column Classification**

In [13]:
id_columns = ['PassengerId']

categorical_columns = [
    'HomePlanet',
    'CryoSleep',
    'Cabin',
    'Destination',
    'VIP',
    'Name'
]

numerical_columns = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

target_column = 'Transported'

print("ID Columns:", id_columns)
print("Categorical Columns:", categorical_columns)
print("Numerical Columns:", numerical_columns)
print("Target Column:", target_column)

ID Columns: ['PassengerId']
Categorical Columns: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'Name']
Numerical Columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Target Column: Transported


# **Step 9 — Data Cleaning**

In [14]:
train.duplicated().sum()

np.int64(0)

In [15]:
test.duplicated().sum()

np.int64(0)

# **Step 10 — Check Unique Values**

In [16]:
train.nunique().sort_values()

CryoSleep          2
VIP                2
Transported        2
Destination        3
HomePlanet         3
Age               80
ShoppingMall    1115
RoomService     1273
VRDeck          1306
Spa             1327
FoodCourt       1507
Cabin           6560
Name            8473
PassengerId     8693
dtype: int64

# **Step 11 — Check Class Balance**

In [17]:
train['Transported'].value_counts(normalize=True) * 100

Transported
True     50.362361
False    49.637639
Name: proportion, dtype: float64

# **Step 12 — Missing Values Percentage**

In [18]:
(train.isnull().sum() / len(train) * 100).sort_values(ascending=False)

CryoSleep       2.496261
ShoppingMall    2.392730
VIP             2.335212
HomePlanet      2.312205
Name            2.300702
Cabin           2.289198
VRDeck          2.162660
Spa             2.105142
FoodCourt       2.105142
Destination     2.093639
RoomService     2.082135
Age             2.059128
PassengerId     0.000000
Transported     0.000000
dtype: float64

# **Step 13 — Start Data Cleaning**

In [19]:
train_clean = train.copy()
test_clean = test.copy()

In [20]:
train_clean.shape, test_clean.shape

((8693, 14), (4277, 13))

# **Step 14 — Missing Values Report**

In [21]:
missing_train = train_clean.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

print(missing_train)

CryoSleep       217
ShoppingMall    208
VIP             203
HomePlanet      201
Name            200
Cabin           199
VRDeck          188
FoodCourt       183
Spa             183
Destination     182
RoomService     181
Age             179
dtype: int64


# **Step 15 — First Missing Value Handling**

In [22]:
train_clean['Age'] = train_clean['Age'].fillna(train_clean['Age'].median())

test_clean['Age'] = test_clean['Age'].fillna(test_clean['Age'].median())

In [23]:
print(train_clean['Age'].isnull().sum())
print(test_clean['Age'].isnull().sum())

0
0


# **Step 16 — HomePlanet Missing Values**

In [24]:
train_clean['HomePlanet'] = train_clean['HomePlanet'].fillna(
    train_clean['HomePlanet'].mode()[0]
)

test_clean['HomePlanet'] = test_clean['HomePlanet'].fillna(
    test_clean['HomePlanet'].mode()[0]
)

In [25]:
print(train_clean['HomePlanet'].isnull().sum())
print(test_clean['HomePlanet'].isnull().sum())

0
0


# **Step 17 — Destination Missing Values**

In [26]:
train_clean['Destination'] = train_clean['Destination'].fillna(
    train_clean['Destination'].mode()[0]
)

test_clean['Destination'] = test_clean['Destination'].fillna(
    test_clean['Destination'].mode()[0]
)

In [27]:
print(train_clean['Destination'].isnull().sum())
print(test_clean['Destination'].isnull().sum())

0
0


# **Step 18 — CryoSleep Missing Values**

In [28]:
train_clean['CryoSleep'] = train_clean['CryoSleep'].fillna(
    train_clean['CryoSleep'].mode()[0]
)

test_clean['CryoSleep'] = test_clean['CryoSleep'].fillna(
    test_clean['CryoSleep'].mode()[0]
)

/tmp/ipykernel_16/4278378405.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_clean['CryoSleep'] = train_clean['CryoSleep'].fillna(
/tmp/ipykernel_16/4278378405.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_clean['CryoSleep'] = test_clean['CryoSleep'].fillna(


In [29]:
print(train_clean['CryoSleep'].isnull().sum())
print(test_clean['CryoSleep'].isnull().sum())

0
0


# **Step 19 — VIP Missing Values**

In [30]:
train_clean['VIP'] = (
    train_clean['VIP']
    .fillna(train_clean['VIP'].mode()[0])
    .infer_objects(copy=False)
)

test_clean['VIP'] = (
    test_clean['VIP']
    .fillna(test_clean['VIP'].mode()[0])
    .infer_objects(copy=False)
)

/tmp/ipykernel_16/18625380.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(train_clean['VIP'].mode()[0])
/tmp/ipykernel_16/18625380.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(test_clean['VIP'].mode()[0])


In [31]:
print(train_clean['VIP'].isnull().sum())
print(test_clean['VIP'].isnull().sum())

0
0


# **Step 19 — Numerical Columns Missing Values Handle**

In [32]:
train_clean['RoomService'] = train_clean['RoomService'].fillna(
    train_clean['RoomService'].median()
)

test_clean['RoomService'] = test_clean['RoomService'].fillna(
    test_clean['RoomService'].median()
)

In [33]:
print(train_clean['RoomService'].isnull().sum())
print(test_clean['RoomService'].isnull().sum())

0
0


# **Step 20 — FoodCourt Missing Values**

In [34]:
train_clean['FoodCourt'] = train_clean['FoodCourt'].fillna(
    train_clean['FoodCourt'].median()
)

test_clean['FoodCourt'] = test_clean['FoodCourt'].fillna(
    test_clean['FoodCourt'].median()
)

In [35]:
print(train_clean['FoodCourt'].isnull().sum())
print(test_clean['FoodCourt'].isnull().sum())

0
0


# **Step 22 — Spa Missing Values**

In [36]:
train_clean['Spa'] = train_clean['Spa'].fillna(
    train_clean['Spa'].median()
)

test_clean['Spa'] = test_clean['Spa'].fillna(
    test_clean['Spa'].median()
)

In [37]:
print(train_clean['Spa'].isnull().sum())
print(test_clean['Spa'].isnull().sum())

0
0


# **Step 23 — VRDeck Missing Values**

In [38]:
train_clean['VRDeck'] = train_clean['VRDeck'].fillna(
    train_clean['VRDeck'].median()
)

test_clean['VRDeck'] = test_clean['VRDeck'].fillna(
    test_clean['VRDeck'].median()
)

In [39]:
print(train_clean['VRDeck'].isnull().sum())
print(test_clean['VRDeck'].isnull().sum())

0
0


# **Step 24 — Check Remaining Missing Values**

In [40]:
train_clean.isnull().sum().sort_values(ascending=False)

ShoppingMall    208
Name            200
Cabin           199
CryoSleep         0
HomePlanet        0
PassengerId       0
Age               0
Destination       0
RoomService       0
VIP               0
FoodCourt         0
Spa               0
VRDeck            0
Transported       0
dtype: int64

In [41]:
test_clean.isnull().sum().sort_values(ascending=False)

Cabin           100
ShoppingMall     98
Name             94
PassengerId       0
HomePlanet        0
Destination       0
CryoSleep         0
Age               0
VIP               0
FoodCourt         0
RoomService       0
Spa               0
VRDeck            0
dtype: int64

In [42]:
print(train_clean['ShoppingMall'].isnull().sum())
print(test_clean['ShoppingMall'].isnull().sum())

208
98


In [43]:
train_clean['ShoppingMall'] = train_clean['ShoppingMall'].fillna(
    train_clean['ShoppingMall'].median()
)

test_clean['ShoppingMall'] = test_clean['ShoppingMall'].fillna(
    test_clean['ShoppingMall'].median()
)

In [44]:
print(train_clean['ShoppingMall'].isnull().sum())
print(test_clean['ShoppingMall'].isnull().sum())

0
0


# **Step 24 (Re-Verification)**

In [45]:
train_clean.isnull().sum().sort_values(ascending=False)

Name            200
Cabin           199
HomePlanet        0
CryoSleep         0
Destination       0
PassengerId       0
Age               0
VIP               0
FoodCourt         0
RoomService       0
ShoppingMall      0
Spa               0
VRDeck            0
Transported       0
dtype: int64

In [46]:
test_clean.isnull().sum().sort_values(ascending=False)

Cabin           100
Name             94
PassengerId       0
CryoSleep         0
HomePlanet        0
Destination       0
Age               0
RoomService       0
VIP               0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
dtype: int64

# **Step 25 — Cabin Missing Values**

In [47]:
train_clean['Cabin'] = train_clean['Cabin'].fillna('Unknown')

test_clean['Cabin'] = test_clean['Cabin'].fillna('Unknown')

In [48]:
print(train_clean['Cabin'].isnull().sum())
print(test_clean['Cabin'].isnull().sum())

0
0


In [49]:
# Fill missing values in Cabin
train_clean['Cabin'] = train_clean['Cabin'].fillna('Unknown')
test_clean['Cabin'] = test_clean['Cabin'].fillna('Unknown')

# Fill missing values in Name
train_clean['Name'] = train_clean['Name'].fillna('Unknown')
test_clean['Name'] = test_clean['Name'].fillna('Unknown')

In [50]:
print(train_clean.isnull().sum().sort_values(ascending=False))
print()
print(test_clean.isnull().sum().sort_values(ascending=False))

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
dtype: int64


In [51]:
# Split Cabin into Deck, CabinNumber, and Side

train_clean[['Deck', 'CabinNumber', 'Side']] = train_clean['Cabin'].str.split('/', expand=True)

test_clean[['Deck', 'CabinNumber', 'Side']] = test_clean['Cabin'].str.split('/', expand=True)

In [52]:
train_clean[['Cabin', 'Deck', 'CabinNumber', 'Side']].head()

,Cabin,Deck,CabinNumber,Side
0,B/0/P,B,0,P
1,F/0/S,F,0,S
2,A/0/S,A,0,S
3,A/0/S,A,0,S
4,F/1/S,F,1,S


In [53]:
test_clean[['Cabin', 'Deck', 'CabinNumber', 'Side']].head()

,Cabin,Deck,CabinNumber,Side
0,G/3/S,G,3,S
1,F/4/S,F,4,S
2,C/0/S,C,0,S
3,C/1/S,C,1,S
4,F/5/S,F,5,S
